# <h1 style="text-align: center;">Optimal Control of TCLab using a Gaussian process regression embedded in Pyomo - notebook v6 with v5c2</h1>

<p style="text-align: center;">Alex Dowling<sup>a</sup>, Jacob P. Krell<sup>b</sup>, David S. Mebane<sup>b</sup>

<p style="text-align: center;"><sup>a</sup>Department of Chemical and Biomolecular Engineering, University of Notre Dame, Notre Dame, IN 46556, USA <br>
<sup>b</sup>Department of Mechanical and Aerospace Engineering, West Virginia University, Morgantown, WV, 26506-6106, USA</p>

## Change Log
- This files is derived from `pyomo_tclab_v5c2.ipynb`
- First implementation of optimization (derived from `pyomo_tclab_v6_optimize1.ipynb`) in Pyomo

## Setup

In [ ]:
import os
dir = os.path.abspath('')  # directory of notebook
import pandas as pd
import numpy as np
from FoKL import FoKLRoutines
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

Load and parse data:

In [ ]:
data = pd.read_csv(os.path.join(dir, "tclab_sine_test.csv"))

tvec = data["Time"].values
Q1 = data["Q1"].values
TS1 = data["T1"].values

Define heater power control signal:

In [ ]:
Q1f = interp1d(tvec, Q1, kind='previous')  # piecewise Q1
dQ1f_analytic = lambda t: 1500 * np.cos(30 * np.pi * t / tvec[-1]) * np.pi / tvec[-1]  # derivative of analytic Q1
dQ1f = interp1d(tvec, dQ1f_analytic(tvec), kind='previous')  # piecewise derivative of analytic Q1

## Derivative of Smoothed Data

Since raw measurements are noisy, a smoothing functions is applied before calculating the time derivative.

### Smoothing

Using a rolling average as the smoothing function,

In [ ]:
window = 9  # odd number, mean at center +/- floor(window/2))

In [ ]:
def smooth(TS1, window):
    """Apply centered average of size window."""
    TS1_smooth = np.zeros_like(TS1)
    w2 = int(np.floor(window / 2))
    w2p1 = w2 + 1

    # bleed in:
    for i in range(w2):
        TS1_smooth[i] = np.mean(TS1[:(i + w2p1)])

    # center:
    for i in range(w2, TS1_smooth.size - w2):
        TS1_smooth[i] = np.mean(TS1[(i - w2):(i + w2p1)])

    # bleed out:
    for i in range(-w2, 0):
        TS1_smooth[i] = np.mean(TS1[(i - w2)::])

    return TS1_smooth

TS1_smooth = smooth(TS1, window)

### Derivative

In [ ]:
def gradient_h4(x, h):
    """h is step size. Order of error is h^4."""
    dx = np.zeros_like(x)

    # bleed in:
    h2 = 2 * h
    dx[0] = (x[1] - x[0]) / h
    dx[1] = (x[2] - x[0]) / h2

    # center difference:
    h12 = 12 * h
    for i in range(2, x.shape[0] - 2):
        dx[i] = (x[i - 2] - 8 * x[i - 1] + 8 * x[i + 1] - x[i + 2]) / h12
    
    # bleed out:
    dx[-2] = (x[-1] - x[-3]) / h2
    dx[-1] = (x[-1] - x[-2]) / h

    return dx

dTS1 = gradient_h4(TS1_smooth, tvec[1] - tvec[0])
dTS1f = interp1d(tvec, dTS1, kind='previous')  # piecewise, grab previous value

## GP Model of Derivative

In [ ]:
GP_dT = FoKLRoutines.FoKL(kernel=1, UserWarnings=False, aic=True)
_ = GP_dT.fit([TS1_smooth, Q1, dQ1f(tvec)], dTS1, clean=True)

In [ ]:
dTS1_GP = GP_dT.evaluate()

%matplotlib inline
plt.figure()
plt.plot(tvec, dTS1)
plt.plot(tvec, dTS1_GP)
plt.title("GP Model of Derivative")
plt.xlabel('Time (s)')
plt.ylabel('Temperature / Time (°C/s)')
plt.legend(['Training Data', 'GP Model'])
plt.grid()

Validation of GP model:

In [ ]:
dt = tvec[1] - tvec[0]  # assume constant time step

def dy_GP(t, y):
    """ODE to integrate GP of derivative."""
    return [GP_dT.evaluate([y[0], Q1f(t), dQ1f(t)], clean=True, SingleInstance=True)[0]]

soln_GP = solve_ivp(dy_GP, [tvec[0], tvec[-1]], [TS1_smooth[0]], 'LSODA', tvec, first_step=1, min_step=1, max_step=1)

In [ ]:
TS1_GP = soln_GP.y[0]

%matplotlib inline
plt.figure()
plt.plot(tvec, TS1_training)
plt.plot(tvec, TS1_GP)
plt.title('Integral of Derivative')
plt.xlabel('Time (s)')
plt.ylabel('Temperature (°C)')
plt.legend(['via Training Data', 'via GP Model'])
plt.grid()

## Application of Validated Dynamics $\dot{T} = f(T, Q, \dot{Q})$ to Step Test

### Setup

In [ ]:
data_test = pd.read_csv(os.path.join(dir, "tclab_step_test.csv"))

tvec_test = data_test["Time"].values
TS1_test = data_test["T1"].values

### Smoothed Data

In [ ]:
TS1_test_smooth = smooth(TS1_test, window)

dTS1_test = gradient_h4(TS1_test_smooth, tvec_test[1] - tvec_test[0])
dTS1f_test = interp1d(tvec_test, dTS1_test, kind='previous')  # piecewise, grab previous value

### Integration of Dynamics from sine-test GP Model

In [ ]:
dt_test = tvec_test[1] - tvec_test[0]  # assume constant time step

def dy_GP_test(t, y):
    """ODE to integrate GP of derivative."""
    return [GP_dT.evaluate([y[0], 50, 0], clean=True, SingleInstance=True)[0]]  # for step, (Q1 = 50, dQ1 = 0)

soln_GP_test = solve_ivp(dy_GP_test, [tvec_test[0], tvec_test[-1]], [TS1_test_smooth[0]], 'LSODA', tvec_test, first_step=1, min_step=1, max_step=1)

### Benchmark Comparison

In [ ]:
%matplotlib inline
plt.figure()
plt.plot(tvec_test, TS1_test)
plt.plot(tvec_test, TS1_benchmark_test)
plt.plot(tvec_test, TS1_test_GP)
plt.title('Benchmark Comparison, Step Test')
plt.xlabel('Time (s)')
plt.ylabel('Temperature (°C)')
plt.legend(['Measured', 'Benchmark', 'via GP Model'])
plt.grid()
plt.show()

RMSE_benchmark_test = _rmse(TS1_test, TS1_benchmark_test)
RMSE_GP_test = _rmse(TS1_test, TS1_test_GP)

print(f"\
| Method       | RMSE |\n\
|--------------|------|\n\
| Benchmark    | {"{0:0.2f}".format(round(RMSE_benchmark_test, 2))} |\n\
| via GP Model | {"{0:0.2f}".format(round(RMSE_GP_test, 2))} |")

---
---
---
---
---
---

# ODE Pyomo Stuff:

note above is likely missing some stuff, and maybe has extraneous stuff

see v6_optimize1 for below, then see v6_optimize1_notes for adding GP

attempt indexing v3.3.0 fokl_to_pyomo variables over ODE time t, then if fails try developing fokl_to_pyomo for continuous set t